In [ ]:
import pandas as pd
import numpy as np

from cdasws import CdasWs, TimeInterval

cdas = CdasWs()
from cdasws.datarepresentation import DataRepresentation as dr

In [ ]:
from pathlib import Path


FILL_VALUES = [
    -1.0e31,
    -9.999999848243207e30,
    -99999.0,
    -999.9000244140625,
]


EARTH_RADIUS_KM = 6378.0
AU_KM = 149597870.7
LAST_VAR_UNITS = {}


def _normalize_unit(unit):
    if unit is None:
        return None
    u = str(unit).strip().lower()
    compact = u.replace(" ", "").replace("_", "")

    if "deg" in compact:
        return "deg"
    if compact == "au" or "astronomical" in compact:
        return "au"
    if "km" in compact:
        return "km"
    if (
        "earthradii" in compact
        or "earthradius" in compact
        or compact == "re"
        or "re" in compact
    ):
        return "re"
    return None


def infer_distance_unit(series, sat_name=""):
    s = pd.to_numeric(series, errors="coerce").dropna()
    if s.empty:
        return "re"

    abs_median = float(np.nanmedian(np.abs(s)))
    sat = sat_name.upper()

    if sat in {"STEREO", "PSP"} and abs_median < 5:
        return "au"
    if abs_median > 1000:
        return "km"
    return "re"


def _unit_to_re_factor(unit):
    if unit == "re":
        return 1.0
    if unit == "km":
        return 1.0 / EARTH_RADIUS_KM
    if unit == "au":
        return AU_KM / EARTH_RADIUS_KM
    raise ValueError(f"Unsupported distance unit: {unit}")


def coerce_distance_cols_to_re(df, cols, sat_name, unit_hint=None, default_unit=None):
    out = df.copy()
    hint = _normalize_unit(unit_hint) or _normalize_unit(default_unit)

    for col in cols:
        if col not in out.columns:
            continue

        chosen = hint or infer_distance_unit(out[col], sat_name=sat_name)
        if chosen == "deg":
            print(f"[{sat_name}] {col}: angle units detected ({unit_hint!r}), skipped.")
            continue

        factor = _unit_to_re_factor(chosen)
        numeric = pd.to_numeric(out[col], errors="coerce")

        if numeric.notna().any():
            med_before = float(np.nanmedian(np.abs(numeric)))
        else:
            med_before = np.nan

        out[col] = numeric * factor

        if out[col].notna().any():
            med_after = float(np.nanmedian(np.abs(out[col])))
        else:
            med_after = np.nan

        print(
            f"[{sat_name}] {col}: unit={chosen} -> R_e (x{factor:.9g}); "
            f"|median| {med_before:.6g} -> {med_after:.6g}"
        )

    return out


def clamp_bad_values(df, velocity_limit=1e5, magnetic_limit=1e5):
    cleaned = df.replace(FILL_VALUES, np.nan).copy()

    # Explicit B cleanup for Wind magnitude fill leaks
    if "B" in cleaned.columns:
        b = pd.to_numeric(cleaned["B"], errors="coerce")
        b = b.replace([1.7320508e31, -1.7320508e31], np.nan)
        cleaned["B"] = b.mask(b.abs() > magnetic_limit)

    for col in cleaned.select_dtypes(include=[np.number]).columns:
        name = col.lower()

        is_velocity = (
            name == "v"
            or name.startswith("v_")
            or "velocity" in name
            or "vx" in name
            or "vy" in name
            or "vz" in name
        )

        is_magnetic = (
            name == "b"
            or name.startswith("b_")
            or name in {"bt", "bx", "by", "bz"}
            or "magnetic" in name
        )

        if is_velocity:
            cleaned[col] = cleaned[col].mask(cleaned[col].abs() > velocity_limit)

        if is_magnetic:
            cleaned[col] = cleaned[col].mask(cleaned[col].abs() > magnetic_limit)

    return cleaned


def save_parquet(merged, sat, tShock, base_dir="Data"):
    shock_label = str(tShock.tz_localize(None)).replace(":", "-")
    out_dir = Path(base_dir) / shock_label
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / (sat + ".parquet")
    merged.to_parquet(out_path)
    return out_path

# Generic Helper Functions

This notebook uses helper functions for working with CDAS data:

1. **`list_datasets(observatory_group, instrument_type)`** - Lists all available datasets
2. **`download_and_unpack(datasets, dataset_index, time_interval, verbose=True)`** - Downloads and unpacks data to DataFrame
3. **`save_parquet(merged, sat, tShock, base_dir)`** - Saves merged data to parquet
4. **`unpack(data)`** - Unpacks CDAS VarCopy dictionary into DataFrame

**Example:**
```python
datasets = list_datasets("Wind", "Magnetic Fields (space)")
df = download_and_unpack(datasets, 6, time_interval)
```

In [ ]:
def remove_spikes(series, threshold=100):
    """
    Простое удаление выбросов и изолированных точек.

    series: pd.Series или 1D np.array
    threshold: минимальная разница для удаления выброса
    """
    s = series.copy().astype(float)
    n = len(s)

    for i in range(n):
        curr = s[i]
        if np.isnan(curr):
            continue

        # соседние значения
        prev = s[i - 1] if i > 0 else np.nan
        next_ = s[i + 1] if i < n - 1 else np.nan

        # 1) изолированная точка
        if (np.isnan(prev) or prev == np.nan) and (np.isnan(next_) or next_ == np.nan):
            s[i] = np.nan
            continue

        # 2) резкий выброс
        neighbors = [x for x in [prev, next_] if not np.isnan(x)]
        if neighbors:
            if all(abs(curr - x) > threshold for x in neighbors):
                s[i] = np.nan

    return s

In [ ]:
def list_datasets(observatory_group, instrument_type):
    """
    List available datasets for a given observatory and instrument type.

    Args:
        observatory_group: Observatory name (e.g., "Wind", "STEREO", "ACE", "THEMIS")
        instrument_type: Type of instrument (e.g., "Magnetic Fields (space)",
                        "Plasma and Solar Wind", "Particles (space)")

    Returns:
        List of datasets
    """
    datasets = cdas.get_datasets(
        observatoryGroup=observatory_group, instrumentType=instrument_type
    )
    print(f"\n=== {observatory_group} - {instrument_type} ===")
    for index, dataset in enumerate(datasets):
        dataset_id = dataset["Id"]
        dataset_label = dataset["Label"]
        print(f"{index}. {dataset_id}, {dataset_label}")
    return datasets


def download_and_unpack(datasets, dataset_index, time_interval, verbose=True):
    """
    Download and unpack a dataset from CDAS.

    Args:
        datasets: List of datasets from list_datasets()
        dataset_index: Index of the dataset to download
        time_interval: TimeInterval object for the data range
        verbose: Print variable information (default: True)

    Returns:
        DataFrame with unpacked data
    """
    global LAST_VAR_UNITS

    dataset = datasets[dataset_index]
    dataset_doi = dataset.get("Doi") or dataset.get("Id")

    # Get variables
    variables = cdas.get_variables(dataset_doi)
    var_names = []

    LAST_VAR_UNITS = {}

    if verbose:
        print(f"\n=== Variables for {dataset['Id']} ===")
        print(f"Time Interval: {dataset.get('TimeInterval', 'N/A')}")

    for index, variable in enumerate(variables):
        name = variable["Name"]
        var_names.append(name)
        unit = (
            variable.get("Units")
            or variable.get("Unit")
            or variable.get("UNITS")
            or variable.get("units")
        )
        LAST_VAR_UNITS[name] = unit
        if verbose:
            description = variable.get("LongDescription", "")
            print(f"{index}. {name}, {description} [units: {unit}]")

    # Download data
    _, data = cdas.get_data(
        dataset_doi, var_names, time_interval, dataRepresentation=dr.SPACEPY
    )

    # Unpack to DataFrame
    df = unpack(data)
    return df

In [ ]:
def unpack(data_b):
    """
    Преобразует словарь VarCopy из Wind/CDAS в DataFrame.

    Правила:
    - выбирается максимальная длина временного ряда;
    - короткие переменные дополняются NaN;
    - многомерные массивы (векторы) кладутся списком в каждую строку;
    - скаляры и 1D массивы растягиваются до max_len.

    Печатает список финальных колонок.
    """

    # 1. Находим максимальную длину
    lengths = []
    for key in data_b.keys():
        arr = data_b[key][...]
        try:
            lengths.append(len(arr))
        except TypeError:
            lengths.append(1)  # 0-D массивы считаем длиной 1

    max_len = max(lengths)
    print(f"Максимальная длина временного ряда: {max_len}")

    # 2. Преобразуем данные для DataFrame
    df_dict = {}

    for key in data_b.keys():
        arr = np.asarray(data_b[key][...])

        if arr.ndim == 0:
            # 0-D скаляр → повторяем
            df_dict[key] = [arr.item()] * max_len
        elif arr.ndim == 1:
            # 1D массив → дополняем NaN
            padded = np.full(max_len, np.nan, dtype=object)
            padded[: len(arr)] = arr
            df_dict[key] = padded
        else:
            # многомерный массив → список в каждой строке, оставшиеся строки None
            padded = [None] * max_len
            for i in range(len(arr)):
                padded[i] = arr[i].tolist()
            df_dict[key] = padded

    # 3. Создаём DataFrame
    df = pd.DataFrame(df_dict)

    # 4. Печатаем финальные колонки
    print("Финальные колонки:", df.columns.tolist())

    return df

# Dates

In [ ]:
import pandas as pd

tShock = pd.to_datetime("2022-11-24T19:10:00Z")

t2 = tShock + pd.Timedelta(hours=6)
t1 = tShock - pd.Timedelta(hours=6)

In [ ]:
time_interval = TimeInterval(t1, t2)
t_start = pd.to_datetime(t1)
t_end = pd.to_datetime(t2)

# Wind

## B3GSE

In [ ]:
# List and download Wind OMNI data for DST
datasets = list_datasets("Wind", "Magnetic Fields (space)")

# выбрать в datasets[N]["Doi"] номер N из списка выше, который нужно скачать---- 0 НАДО для DSt
N = 0
print("info---------", N, "---", datasets[N]["TimeInterval"])  # - инфо о данных :р
df = download_and_unpack(datasets, N, time_interval)

In [ ]:
DST_omni = df.filter(["Epoch", "DST"], axis=1)
DST_omni = DST_omni.set_index("Epoch")

In [ ]:
DST_omni.plot(title="DST from Wind OMNI data", ylabel="DST, nT")

In [ ]:
# List available datasets for Wind magnetic field
datasets = list_datasets("Wind", "Magnetic Fields (space)")

In [ ]:
print(datasets[6]["TimeInterval"])

In [ ]:
# Download Wind magnetic field data (3 sec resolution)
N = 6
df = download_and_unpack(datasets, N, time_interval)

In [ ]:
# магнитное поле 3 sec
Wind_MFI = df.filter(["Epoch3", "B3GSE"], axis=1)
Wind_MFI = Wind_MFI.set_index("Epoch3")
Wind_MFI.dropna(inplace=True)
# распаковка списка bx by bz в отдельные столбцы
Wind_MFI[["B_X_GSE", "B_Y_GSE", "B_Z_GSE"]] = pd.DataFrame(
    Wind_MFI["B3GSE"].tolist(), index=Wind_MFI.index
)
Wind_MFI = Wind_MFI.drop(["B3GSE"], axis=1)
Wind_MFI["B"] = Wind_MFI.apply(
    lambda y: np.sqrt(y["B_X_GSE"] ** 2 + y["B_Y_GSE"] ** 2 + y["B_Z_GSE"] ** 2), axis=1
)
# фильтр плохих данных
Wind_MFI = clamp_bad_values(Wind_MFI)
Wind_MFI.info()
# усреднение 1 час начиная с 00
# df_MFI_1h_edit=df_MFI_1h.resample('1h').mean()

In [ ]:
Wind_MFI.plot()

## PGSE

In [ ]:
# магнитное поле 1 min
Wind_PGSE = df.filter(["Epoch", "PGSE"], axis=1)
Wind_PGSE = Wind_PGSE.set_index("Epoch")
Wind_PGSE.dropna(inplace=True)
Wind_PGSE

In [ ]:
# распаковка списка bx by bz в отдельные столбцы
Wind_PGSE[["X_GSE", "Y_GSE", "Z_GSE"]] = pd.DataFrame(
    Wind_PGSE["PGSE"].tolist(), index=Wind_PGSE.index
)
Wind_PGSE = Wind_PGSE.drop(["PGSE"], axis=1)
# фильтр плохих данных
Wind_PGSE = clamp_bad_values(Wind_PGSE)
Wind_PGSE = coerce_distance_cols_to_re(
    Wind_PGSE,
    ["X_GSE", "Y_GSE", "Z_GSE"],
    sat_name="Wind",
    unit_hint=LAST_VAR_UNITS.get("PGSE"),
    default_unit="re",
)
Wind_PGSE.info()
# усреднение 1 час начиная с 00
# df_PGSE_1h_edit=df_PGSE_1h.resample('1h').mean()

## SWE

In [ ]:
# List available datasets for Wind plasma
datasets = list_datasets("Wind", "Plasma and Solar Wind")

# Download Wind SWE data
N = 11  # WI_K0_SWE
df = download_and_unpack(datasets, N, time_interval)

In [ ]:
datasets[11]["TimeInterval"]

In [ ]:
# СВ
Wind_SW = df.filter(
    [
        "Epoch",
        "Proton_V_moment",
        "Proton_VX_moment",
        "Proton_VY_moment",
        "Proton_VZ_moment",
        "Proton_Np_nonlin",
    ],
    axis=1,
)
Wind_SW = Wind_SW.rename(
    columns={
        "Proton_V_moment": "V",
        "Proton_VX_moment": "V_X_GSE",
        "Proton_VY_moment": "V_Y_GSE",
        "Proton_VZ_moment": "V_Z_GSE",
        "Proton_Np_nonlin": "N_p",
    }
)
Wind_SW

In [ ]:
Wind_SW = Wind_SW.set_index("Epoch")
Wind_SW.dropna(inplace=True)


# Wind_SW = Wind_SW.drop(["V_GSE"], axis=1)
# Wind_SW["v"] = Wind_SW.apply(
#     lambda y: np.sqrt(y["v_x"] ** 2 + y["v_y"] ** 2 + y["v_z"] ** 2), axis=1
# )
# фильтр плохих данных (удаляет всю строку)
Wind_SW = clamp_bad_values(Wind_SW)
Wind_SW.info()

# усреднение 1 час начиная с 00
# df_SW_1h_edit=df_SW_1h.resample('1h').mean()

# 1 день - 7 секунд

In [ ]:
# Wind_SW_cleaned = Wind_SW.copy()
# Wind_SW_cleaned["N_p"] = remove_spikes(
#     Wind_SW["N_p"], threshold=15
# )
# Wind_SW_cleaned["Proton_V_nonlin"] = remove_spikes(
#     Wind_SW["Proton_V_nonlin"], threshold=100
# )

# Wind_SW_cleaned["Proton_V_nonlin"] = Wind_SW_cleaned["Proton_V_nonlin"].loc[
#     Wind_SW_cleaned["Proton_V_nonlin"] < 600
# ]

# Wind_SW_cleaned["V_X_GSE"] = remove_spikes(
#     Wind_SW["V_X_GSE"], threshold=100
# )
# Wind_SW_cleaned["V_Y_GSE"] = remove_spikes(
#     Wind_SW["V_Y_GSE"], threshold=100
# )
# #.loc[Wind_SW_cleaned["V_Y_GSE"] < -600]

# Wind_SW_cleaned["V_Z_GSE"] = remove_spikes(
#     Wind_SW["V_Z_GSE"], threshold=100
# )
Wind_SW.plot()
# Wind_SW_cleaned.plot()

## Plot and Export

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

In [ ]:
# 1. Определяем общий интервал
t_start = max(Wind_SW.index.min(), Wind_MFI.index.min())
t_end = min(Wind_SW.index.max(), Wind_MFI.index.max())
print(f"Общий диапазон: {t_start} — {t_end}")

# 2. Обрезаем данные по интервалу
SW = Wind_SW[(Wind_SW.index >= t_start) & (Wind_SW.index <= t_end)]
MFI = Wind_MFI[(Wind_MFI.index >= t_start) & (Wind_MFI.index <= t_end)]

t = (
    SW.index
)  # используем индекс SW для построения, можно интерполяцией привести MFI к этому индексу

# --- Теперь строим график как раньше ---
fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)
plt.subplots_adjust(hspace=0.3)

# Верхняя панель: поле
ax = axes[0]
ax.plot(MFI.index, MFI["B"], "k", label="|B|")
ax.plot(MFI.index, MFI["B_X_GSE"], "r", label="B_x")
ax.plot(MFI.index, MFI["B_Y_GSE"], "g", label="B_y")
ax.plot(MFI.index, MFI["B_Z_GSE"], "b", label="B_z")
ax.set_ylabel("B [nT]")
ax.legend(loc="upper right")
ax.grid(True)
ax.set_xlim(t_start, t_end)  # единый масштаб

# Средняя панель: V и Np
ax = axes[1]

ax.plot(SW.index, abs(SW["V_X_GSE"]), "r", label="|V_x|")
ax.plot(SW.index, SW["V"], "k", label="|V|")
ax.set_ylabel("V [km/s]")
ax.grid(True)
ax.legend(loc="upper left")

ax2 = ax.twinx()
ax2.plot(SW.index, SW["N_p"], "b", label="Np", linestyle="--")
ax2.set_ylabel("Np [cm^-3]")
ax2.legend(loc="upper right")

ax.set_xlim(t_start, t_end)

# Нижняя панель: Vy и Vz
ax = axes[2]
ax.plot(SW.index, SW["V_Y_GSE"], "g", label="V_y")
SW_cl = SW.loc[SW["V_Z_GSE"] > -600]
ax.plot(SW_cl.index, SW_cl["V_Z_GSE"], "b", label="V_z")
ax.set_ylabel("V_y, V_z [km/s]")
ax.set_xlabel("Time")
ax.legend(loc="upper right")
ax.grid(True)
ax.set_xlim(t_start, t_end)

# Форматирование времени
# axes[-1].xaxis.set_major_locator(mdates.DayLocator())
# axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%d-%b"))
# axes[-1].xaxis.set_minor_locator(mdates.HourLocator(byhour=[0, 6, 12, 18]))
# plt.setp(axes[-1].xaxis.get_majorticklabels(), rotation=30, ha="right")

for ax in axes:
    ax.grid(which="minor", linestyle="--", color="lightgrey", alpha=0.5)

plt.show()

In [ ]:
mfi = Wind_MFI.copy()
sw = Wind_SW.copy()
coord = Wind_PGSE.copy()

# Normalize index names and to UTC datetime index
for df in (mfi, sw, coord):
    df.index = pd.to_datetime(df.index, utc=True)
    df.index.name = "Epoch"

# Replace CDAS fill values (e.g., -1e31) with NaN
mfi = clamp_bad_values(mfi)
sw = clamp_bad_values(sw)
coord = clamp_bad_values(coord)

# Pick a "unified" epoch grid — here we use MFI timestamps
# You can swap to sw.index or coord.index if preferred.
epoch = mfi.index

# Nearest-time merge with tolerance (e.g., 2 minutes)
tol = pd.Timedelta("2min")

mfi_u = mfi.reset_index().sort_values("Epoch")
sw_u = sw.reset_index().sort_values("Epoch")
coord_u = coord.reset_index().sort_values("Epoch")

merged = pd.merge_asof(
    mfi_u,
    sw_u,
    on="Epoch",
    direction="nearest",
    tolerance=tol,
    suffixes=("_mfi", "_sw"),
)
merged = pd.merge_asof(merged, coord_u, on="Epoch", direction="nearest", tolerance=tol)

merged = merged.set_index("Epoch").sort_index()

mfi_cols = list(mfi.columns)
sw_cols = list(sw.columns)

merged = merged.dropna(subset=mfi_cols + sw_cols, how="all")

In [ ]:
merged

In [ ]:
save_parquet(merged, "Wind", tShock)

# STEREO

## Mag

In [ ]:
# List available datasets for STEREO magnetic field
datasets = list_datasets("STEREO", "Magnetic Fields (space)")

# Download STEREO magnetic field data
N = 5
df = download_and_unpack(datasets, N, time_interval)

In [ ]:
Stereo_MFI = df.filter(["Epoch", "BFIELD"], axis=1)
Stereo_MFI = Stereo_MFI.set_index("Epoch")
Stereo_MFI.dropna(inplace=True)

# распаковка списка bx by bz в отдельные столбцы
Stereo_MFI[["B_X_HGE", "B_Y_HGE", "B_Z_HGE", "B"]] = pd.DataFrame(
    Stereo_MFI["BFIELD"].tolist(), index=Stereo_MFI.index
)
Stereo_MFI = Stereo_MFI.drop(["BFIELD"], axis=1)
# Stereo["B"] = Stereo.apply(lambda y: np.sqrt(y["B_X_HGE"] ** 2 + y["B_Y_HGE"] ** 2 + y["B_Z_HGE"] ** 2), axis=1)
# фильтр плохих данных
Stereo_MFI = clamp_bad_values(Stereo_MFI)
Stereo_MFI.info()
# усреднение 1 секунда начиная с 00
Stereo_MFI_1s = Stereo_MFI.resample("1s").mean()

In [ ]:
Stereo_MFI_1s.plot()

## PLASTIC

In [ ]:
# List available datasets for STEREO plasma
datasets = list_datasets("STEREO", "Plasma and Solar Wind")

In [ ]:
# Download STEREO PLASTIC data
N = 10
df = download_and_unpack(datasets, N, time_interval)

In [ ]:
# магнитное поле 1 min
Stereo_SW = df.filter(
    ["epoch", "proton_number_density", "proton_bulk_speed", "proton_temperature"],
    axis=1,
)
Stereo_SW = Stereo_SW.set_index("epoch")
Stereo_SW = Stereo_SW.rename(
    columns={
        "proton_number_density": "N_p",
        "proton_bulk_speed": "V",
        "proton_temperature": "T_p",
    }
)
Stereo_SW_cleaned = Stereo_SW.copy()
Stereo_SW_cleaned["T_p"] = remove_spikes(Stereo_SW_cleaned["T_p"], threshold=10000)
Stereo_SW_cleaned["N_p"] = remove_spikes(Stereo_SW_cleaned["N_p"], threshold=10000)
Stereo_SW_cleaned["V"] = remove_spikes(Stereo_SW_cleaned["V"], threshold=10000)
Stereo_SW_cleaned["T_p"] = Stereo_SW_cleaned["T_p"].replace(
    -9.999999848243207e30, np.nan
)
Stereo_SW_cleaned["N_p"] = Stereo_SW_cleaned["N_p"].replace(
    -9.999999848243207e30, np.nan
)
Stereo_SW_cleaned["V"] = Stereo_SW_cleaned["V"].replace(-9.999999848243207e30, np.nan)
Stereo_SW_cleaned = clamp_bad_values(Stereo_SW_cleaned)
-9.999999848243207e30
# Stereo_SW.dropna(inplace=True)

In [ ]:
Stereo_SW_cleaned.plot()

## Coord

In [ ]:
# List available datasets for STEREO coordinates (reuse plasma list)
datasets = list_datasets("STEREO", "Plasma and Solar Wind")

In [ ]:
# Download STEREO coordinate data
N = 0
df = download_and_unpack(datasets, N, time_interval)

In [ ]:
Stereo_HGE = df.filter(
    ["Epoch", "radialDistance", "heliographicLatitude", "heliographicLongitude"],
)
Stereo_HGE = Stereo_HGE.set_index("Epoch")
Stereo_HGE = coerce_distance_cols_to_re(
    Stereo_HGE,
    ["radialDistance"],
    sat_name="STEREO",
    unit_hint=LAST_VAR_UNITS.get("radialDistance"),
    default_unit="au",
)

## Plot and Export

In [ ]:
# #Построение графиков для Stereo если они в одном датафрейме
# t = Stereo.index  # используем индекс Stereo для построения, можно интерполяцией привести Stereo к этому индексу

# # --- Теперь строим график как раньше ---
# fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)
# plt.subplots_adjust(hspace=0.3)

# # Верхняя панель: поле
# ax = axes[0]
# ax.plot(Stereo.index, Stereo['B'], 'k', label='|B|')
# ax.plot(Stereo.index, Stereo['B_X_HGE'], 'r', label='B_x')
# ax.plot(Stereo.index, Stereo['B_Y_HGE'], 'g', label='B_y')
# ax.plot(Stereo.index, Stereo['B_Z_HGE'], 'b', label='B_z')
# ax.set_ylabel('B [nT]')
# ax.legend(loc='upper right')
# ax.grid(True)
# ax.set_xlim(t_start, t_end)  # единый масштаб

# # Средняя панель: V и Np
# ax = axes[1]

# ax.plot(Stereo.index, Stereo['V'], 'k', label='|V|')
# ax.set_ylabel('V [km/s]')
# ax.grid(True)
# ax.legend(loc='upper left')

# ax2 = ax.twinx()
# ax2.plot(Stereo.index, Stereo['N_p'], 'b', label='Np', linestyle='--')
# ax2.set_ylabel('Np [cm^-3]')
# ax2.legend(loc='upper right')

# ax.set_xlim(t_start, t_end)

# # Нижняя панель: температура
# ax = axes[2]
# ax.plot(Stereo.index, Stereo['T_p'], 'g', label='T')
# ax.set_yscale('log')
# ax.set_ylabel('T [K]')
# ax.set_xlabel('Time')
# ax.legend(loc='upper right')
# ax.grid(True)
# ax.set_xlim(t_start, t_end)

# # Форматирование времени
# axes[-1].xaxis.set_major_locator(mdates.DayLocator())
# axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%d-%b'))
# axes[-1].xaxis.set_minor_locator(mdates.HourLocator(byhour=[0,6,12,18]))
# plt.setp(axes[-1].xaxis.get_majorticklabels(), rotation=30, ha='right')

# for ax in axes:
#     ax.grid(which='minor', linestyle='--', color='lightgrey', alpha=0.5)

# plt.show()

In [ ]:
mfi = Stereo_MFI_1s.copy()
sw = Stereo_SW_cleaned.copy()
coord = Stereo_HGE.copy()

# Normalize index names and to UTC datetime index
for df in (mfi, sw, coord):
    df.index = pd.to_datetime(df.index, utc=True)
    df.index.name = "Epoch"

# Replace CDAS fill values (e.g., -1e31) with NaN
mfi = clamp_bad_values(mfi)
sw = clamp_bad_values(sw)
coord = clamp_bad_values(coord)

# Pick a "unified" epoch grid — here we use MFI timestamps
# You can swap to sw.index or coord.index if preferred.
epoch = mfi.index

# Nearest-time merge with tolerance (e.g., 2 minutes)
tol = pd.Timedelta("2min")

mfi_u = mfi.reset_index().sort_values("Epoch")
sw_u = sw.reset_index().sort_values("Epoch")
coord_u = coord.reset_index().sort_values("Epoch")

merged = pd.merge_asof(
    mfi_u,
    sw_u,
    on="Epoch",
    direction="nearest",
    tolerance=tol,
    suffixes=("_mfi", "_sw"),
)
merged = pd.merge_asof(merged, coord_u, on="Epoch", direction="nearest", tolerance=tol)

merged = merged.set_index("Epoch").sort_index()

mfi_cols = list(mfi.columns)
sw_cols = list(sw.columns)

merged = merged.dropna(subset=mfi_cols + sw_cols, how="all")

In [ ]:
merged

In [ ]:
parquet_path = save_parquet(merged, "STEREO", tShock)

# ACE

## Magnetic Field

In [ ]:
# List available datasets for ACE magnetic field
datasets = list_datasets("ACE", "Magnetic Fields (space)")

# Download ACE magnetic field data
N = 4
df = download_and_unpack(datasets, N, time_interval)

In [ ]:
# магнитное поле 1 min
Ace_MFI = df.filter(["Epoch", "Magnitude", "BGSEc"], axis=1)
Ace_MFI = Ace_MFI.set_index("Epoch")
Ace_MFI.dropna(inplace=True)
# распаковка списка bx by bz в отдельные столбцы
Ace_MFI[["B_X_GSE", "B_Y_GSE", "B_Z_GSE"]] = pd.DataFrame(
    Ace_MFI["BGSEc"].tolist(), index=Ace_MFI.index
)
Ace_MFI = Ace_MFI.drop(["BGSEc"], axis=1)
Ace_MFI["B"] = Ace_MFI.Magnitude
Ace_MFI = Ace_MFI.drop(["Magnitude"], axis=1)
# фильтр плохих данных
Ace_MFI = clamp_bad_values(Ace_MFI)
Ace_MFI.info()
# усреднение 1 час начиная с 00
# df_MFI_1h_edit=df_MFI_1h.resample('1h').mean()
Ace_MFI.plot()

## Solar Wind

In [ ]:
# List available datasets for ACE plasma
datasets = list_datasets("ACE", "Plasma and Solar Wind")

In [ ]:
# Download ACE plasma data
N = 1
df = download_and_unpack(datasets, N, time_interval)

In [ ]:
datasets[N]["TimeInterval"]

In [ ]:
Ace_SW = df.copy()
Ace_SW = Ace_SW.set_index("Epoch")
Ace_SW = Ace_SW.rename(
    columns={
        "Vp": "V",
        "Np": "N_p",
    }
)

if "V_GSE" in Ace_SW.columns:
    Ace_SW[["V_X_GSE", "V_Y_GSE", "V_Z_GSE"]] = pd.DataFrame(
        Ace_SW["V_GSE"].tolist(), index=Ace_SW.index
    )
    Ace_SW = Ace_SW.drop(["V_GSE"], axis=1)

In [ ]:
for col in ["V", "N_p", "V_X_GSE", "V_Y_GSE", "V_Z_GSE"]:
    if col in Ace_SW.columns:
        Ace_SW[col] = Ace_SW[col].replace(-999.9000244140625, np.nan)
Ace_SW = clamp_bad_values(Ace_SW)

In [ ]:
Ace_SW

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# 1. общий интервал
# t_start = max(Stereo.index.min(), Wind_SW_cleaned.index.min())
# t_end   = min(Stereo.index.max(), Wind_SW_cleaned.index.max())
# t_start=pd.to_datetime("2024-06-26" )
# t_end=pd.to_datetime("2024-06-30" )

## Coord

In [ ]:
# List available datasets for ACE coordinates
datasets = list_datasets("ACE", "Magnetic Fields (space)")

In [ ]:
# Download ACE coordinate data
N = 1
df = download_and_unpack(datasets, N, time_interval)

In [ ]:
# магнитное поле 1 min
Ace_GSE = df.filter(["Epoch", "SC_pos_GSE"], axis=1)
Ace_GSE = Ace_GSE.set_index("Epoch")
Ace_GSE.dropna(inplace=True)
# распаковка списка bx by bz в отдельные столбцы
Ace_GSE[["X_GSE", "Y_GSE", "Z_GSE"]] = pd.DataFrame(
    Ace_GSE["SC_pos_GSE"].tolist(), index=Ace_GSE.index
)
Ace_GSE = Ace_GSE.drop(["SC_pos_GSE"], axis=1)
Ace_GSE.info()
# усреднение 1 час начиная с 00
# df_GSE_1h_edit=df_GSE_1h.resample('1h').mean()
Ace_GSE = clamp_bad_values(Ace_GSE)
Ace_GSE = coerce_distance_cols_to_re(
    Ace_GSE,
    ["X_GSE", "Y_GSE", "Z_GSE"],
    sat_name="ACE",
    unit_hint=LAST_VAR_UNITS.get("SC_pos_GSE"),
)
Ace_GSE.plot()

## Plot and Export

In [ ]:
# Crop data to time range
ST_MFI = Stereo_MFI[(Stereo_MFI.index >= t_start) & (Stereo_MFI.index <= t_end)]
ST_SW = Stereo_SW_cleaned[
    (Stereo_SW_cleaned.index >= t_start) & (Stereo_SW_cleaned.index <= t_end)
]
Wind_MFI_cr = Wind_MFI[(Wind_MFI.index >= t_start) & (Wind_MFI.index <= t_end)]
Wind_SW_cr = Wind_SW[(Wind_SW.index >= t_start) & (Wind_SW.index <= t_end)]
DST_omni_cr = DST_omni[(DST_omni.index >= t_start) & (DST_omni.index <= t_end)]
Ace_SW_cr = Ace_SW[(Ace_SW.index >= t_start) & (Ace_SW.index <= t_end)]
Ace_MFI_cr = Ace_MFI[(Ace_MFI.index >= t_start) & (Ace_MFI.index <= t_end)]

# --- Plot ---
fig, axes = plt.subplots(6, 1, figsize=(8, 15), sharex=True)
plt.subplots_adjust(hspace=0.1)

# ---------------- 1. STEREO Magnetic Field ----------------
ax = axes[0]
ax.plot(ST_MFI.index, ST_MFI["B"], "k", label="|B|")
ax.plot(ST_MFI.index, ST_MFI["B_X_HGE"], "r", label="B_x")
ax.plot(ST_MFI.index, ST_MFI["B_Y_HGE"], "g", label="B_y")
ax.plot(ST_MFI.index, ST_MFI["B_Z_HGE"], "b", label="B_z")
ax.set_ylabel("B STEREO [nT]")
ax.legend(loc="upper right")
ax.grid(which="major", linestyle="-", color="grey", alpha=0.7)
ax.grid(which="minor", linestyle="--", color="lightgrey", alpha=0.5)
ax.set_xlim(t_start, t_end)

# ---------------- 2. WIND Magnetic Field ----------------
ax = axes[1]
ax.plot(Wind_MFI_cr.index, Wind_MFI_cr["B"], "k", label="|B|")
ax.plot(Wind_MFI_cr.index, Wind_MFI_cr["B_X_GSE"], "r", label="B_x")
ax.plot(Wind_MFI_cr.index, Wind_MFI_cr["B_Y_GSE"], "g", label="B_y")
ax.plot(Wind_MFI_cr.index, Wind_MFI_cr["B_Z_GSE"], "b", label="B_z")
ax.set_ylabel("B WIND [nT]")
ax.legend(loc="upper right")
ax.grid(which="major", linestyle="-", color="grey", alpha=0.7)
ax.grid(which="minor", linestyle="--", color="lightgrey", alpha=0.5)
ax.set_xlim(t_start, t_end)

# ---------------- 3. ACE Magnetic Field ----------------
ax = axes[2]
ax.plot(Ace_MFI_cr.index, Ace_MFI_cr["B"], "k", label="|B|")
ax.plot(Ace_MFI_cr.index, Ace_MFI_cr["B_X_GSE"], "r", label="B_x")
ax.plot(Ace_MFI_cr.index, Ace_MFI_cr["B_Y_GSE"], "g", label="B_y")
ax.plot(Ace_MFI_cr.index, Ace_MFI_cr["B_Z_GSE"], "b", label="B_z")
ax.set_ylabel("B ACE [nT]")
ax.legend(loc="upper right")
ax.grid(which="major", linestyle="-", color="grey", alpha=0.7)
ax.grid(which="minor", linestyle="--", color="lightgrey", alpha=0.5)
ax.set_xlim(t_start, t_end)

# ---------------- 4. Velocity ----------------
ax = axes[3]
ax.plot(ST_SW.index, ST_SW["V"], "r", label="STEREO |V|")
ax.plot(Wind_SW_cr.index, Wind_SW_cr["V"], "b", label="WIND |V|")
ax.plot(Ace_SW_cr.index, Ace_SW_cr["V"], "g", label="ACE |V|")
ax.set_ylabel("V [km/s]")
ax.legend(loc="upper right")
ax.grid(which="major", linestyle="-", color="grey", alpha=0.7)
ax.grid(which="minor", linestyle="--", color="lightgrey", alpha=0.5)
ax.set_xlim(t_start, t_end)

# ---------------- 5. Density ----------------
ax = axes[4]
ax.plot(ST_SW.index, ST_SW["N_p"], "r", label="STEREO Np")
ax.plot(Wind_SW_cr.index, Wind_SW_cr["N_p"], "b", label="WIND Np")
ax.plot(Ace_SW_cr.index, Ace_SW_cr["N_p"], "g", label="ACE Np")
ax.set_ylabel("Np [cm^-3]")
ax.set_xlabel("Time")
ax.legend(loc="upper right")
ax.grid(which="major", linestyle="-", color="grey", alpha=0.7)
ax.grid(which="minor", linestyle="--", color="lightgrey", alpha=0.5)
ax.set_xlim(t_start, t_end)

# ---------------- 6. Dst ----------------
ax = axes[5]
ax.plot(DST_omni_cr.index, DST_omni_cr["DST"], "k", label="Dst")
ax.set_ylabel("Dst [nT]")
ax.grid(which="major", linestyle="-", color="grey", alpha=0.7)
ax.grid(which="minor", linestyle="--", color="lightgrey", alpha=0.5)
ax.set_xlim(t_start, t_end)

# ---------------- Formatting ----------------
axes[-1].xaxis.set_major_locator(mdates.DayLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%d-%b"))
axes[-1].xaxis.set_minor_locator(mdates.HourLocator(byhour=[0, 6, 12, 18]))
plt.setp(axes[-1].xaxis.get_majorticklabels(), rotation=30, ha="right")
axes[-1].text(
    0.5,
    1.05,
    f"{t_start:%Y}",
    transform=ax.transAxes,
    ha="center",
    va="bottom",
    fontsize=10,
)

plt.show()

In [ ]:
mfi = Ace_MFI.copy()
sw = Ace_SW.copy()
coord = Ace_GSE.copy()

# Normalize index names and to UTC datetime index
for df in (mfi, sw, coord):
    df.index = pd.to_datetime(df.index, utc=True)
    df.index.name = "Epoch"

# Replace CDAS fill values (e.g., -1e31) with NaN
mfi = clamp_bad_values(mfi)
sw = clamp_bad_values(sw)
coord = clamp_bad_values(coord)

# Pick a "unified" epoch grid — here we use MFI timestamps
# You can swap to sw.index or coord.index if preferred.
epoch = mfi.index

# Nearest-time merge with tolerance (e.g., 2 minutes)
tol = pd.Timedelta("2min")

mfi_u = mfi.reset_index().sort_values("Epoch")
sw_u = sw.reset_index().sort_values("Epoch")
coord_u = coord.reset_index().sort_values("Epoch")

merged = pd.merge_asof(
    mfi_u,
    sw_u,
    on="Epoch",
    direction="nearest",
    tolerance=tol,
    suffixes=("_mfi", "_sw"),
)
merged = pd.merge_asof(merged, coord_u, on="Epoch", direction="nearest", tolerance=tol)

merged = merged.set_index("Epoch").sort_index()

mfi_cols = list(mfi.columns)
sw_cols = list(sw.columns)

merged = merged.dropna(subset=mfi_cols + sw_cols, how="all")

In [ ]:
sw_u

In [ ]:
parquet_path = save_parquet(merged, "ACE", tShock)

# DSCOVR


In [ ]:
# List available datasets for DSCOVR ephemeris
datasets = list_datasets("DSCOVR", "Ephemeris/Attitude/Ancillary")

# Download DSCOVR ephemeris data (set N to the ephemeris dataset index from the list above)
N = 2
df = download_and_unpack(datasets, N, time_interval)

time_col = next(
    (c for c in ["Epoch", "epoch", "EPOCH", "time", "Time", "TIME"] if c in df.columns),
    None,
)
vec_col = next((c for c in ["GSE_POS", "gse_pos"] if c in df.columns), None)
if time_col is None or vec_col is None:
    raise ValueError(
        f"Expected time and GSE_POS columns in DSCOVR ephemeris, got: {list(df.columns)}"
    )

dscovr_coord = df.filter([time_col, vec_col], axis=1)
dscovr_coord = dscovr_coord.set_index(time_col)
dscovr_coord.dropna(inplace=True)

dscovr_coord[["X_GSE", "Y_GSE", "Z_GSE"]] = pd.DataFrame(
    dscovr_coord[vec_col].tolist(), index=dscovr_coord.index
)
dscovr_coord = dscovr_coord.drop([vec_col], axis=1)
dscovr_coord = clamp_bad_values(dscovr_coord)
dscovr_coord = coerce_distance_cols_to_re(
    dscovr_coord,
    ["X_GSE", "Y_GSE", "Z_GSE"],
    sat_name="DSCOVR",
    unit_hint=LAST_VAR_UNITS.get(vec_col),
)

dscovr_coord.index = pd.to_datetime(dscovr_coord.index, utc=True)
dscovr_coord.index.name = "time"

dscovr_coord

In [ ]:
#here download data DSCOVR fromngdc.noaa.gov/dscovr/portal/index.html#/download/1748736000000;1748822400000/f3s;m1s;pop and save with initial name (zip unpacked) to DATA/DSCOVR (Raw)/ (create folder)

In [ ]:
import xarray as xr

shock_label = str(tShock.tz_localize(None)).replace(":", "-")
shock_dir = Path("Data") / shock_label
raw_dir = shock_dir / "DSCOVR (Raw)"

fcup_files = sorted(raw_dir.glob("oe_f3s_dscovr_*.nc"))
mag_files = sorted(raw_dir.glob("oe_m1s_dscovr_*.nc"))


fcup_file = fcup_files[-1]
mag_file = mag_files[-1]
print("Using DSCOVR files:")
print(f"  SW:  {fcup_file.name}")
print(f"  MAG: {mag_file.name}")

sw_ds = xr.open_dataset(fcup_file)
mag_ds = xr.open_dataset(mag_file)

sw = sw_ds.to_dataframe()
mag = mag_ds.to_dataframe()

sw_ds.close()
mag_ds.close()

sw.index = pd.to_datetime(sw.index, unit="ms", utc=True)
mag.index = pd.to_datetime(mag.index, unit="ms", utc=True)

In [ ]:
sw = sw[
    [
        "proton_vx_gse",
        "proton_vy_gse",
        "proton_vz_gse",
        "proton_speed",
        "proton_density",
        "proton_temperature",
    ]
].rename(
    columns={
        "proton_vx_gse": "V_X_GSE",
        "proton_vy_gse": "V_Y_GSE",
        "proton_vz_gse": "V_Z_GSE",
        "proton_speed": "V",
        "proton_density": "N_p",
        "proton_temperature": "T",
    }
)

mag = mag[["bx_gse", "by_gse", "bz_gse", "bt"]].rename(
    columns={
        "bx_gse": "B_X_GSE",
        "by_gse": "B_Y_GSE",
        "bz_gse": "B_Z_GSE",
        "bt": "B",
    }
)

sw = clamp_bad_values(sw)
mag = clamp_bad_values(mag)

mfi_u = mag.reset_index().rename(columns={"index": "time"}).sort_values("time")
sw_u = sw.reset_index().rename(columns={"index": "time"}).sort_values("time")

In [ ]:
sw_uu = sw_u.copy()
# sw_u = sw_u.loc[(np.abs(sw_u["V_Y_GSE"]) > 10) & (np.abs(sw_u["V_Z_GSE"]) > 10)]

In [ ]:
sw_u.V_Y_GSE.plot()

In [ ]:
sw_uu.V_Y_GSE.plot()

In [ ]:
coord_u

In [ ]:
merged = pd.merge_asof(
    mfi_u,
    sw_u,
    on="time",
    direction="nearest",
    tolerance=pd.Timedelta("2min"),
    suffixes=("_mfi", "_sw"),
)

coord_u = dscovr_coord.reset_index().sort_values("time")
coord_u.time = coord_u.time.astype('datetime64[ns, UTC]')
merged = pd.merge_asof(
    merged,
    coord_u,
    on="time",
    direction="nearest",
    tolerance=pd.Timedelta("2min"),
)

merged = merged.set_index("time").sort_index()
merged = merged.dropna(subset=list(mag.columns) + list(sw.columns), how="all")

In [ ]:
parquet_path = save_parquet(merged, "DSCOVR", tShock)
parquet_path

In [ ]:
merged

# THEMIS-B

## Velocity

In [162]:
# List available datasets for THEMIS magnetic field
datasets = list_datasets("THEMIS", "Plasma and Solar Wind")


=== THEMIS - Plasma and Solar Wind ===
0. THA_L2_ESA, THEMIS-A (P5): Electrostatic Analyzer (ESA): Electron/Ion Ground-Calculated Energy Fluxes (ions: 5 eV to 25 keV) electrons: 6 eV to 30 keV) and Moments (density, velocity, pressure, and temperature). Includes FULL, REDUCED and BURST modes. FULL: high angular resolution, low (few min) time resolution. REDUCED: degraded angular resolution, high (approx. 3 sec) time resolution. BURST: high angular resolution, high time resolution; only short bursts of data. Note that angular resolution affects moments since they are obtained integrating over the mode-specific angular distribution. - V. Angelopoulos, C.W. Carlson & J. McFadden (UCB, NASA NAS5-02099)
1. THA_L2_MOM, THEMIS-A (P5):  On Board moments:  Electron/Ion moments density, flux, velocity, pressure and temperature. - V. Angelopoulos, C.W. Carlson & J. McFadden (UCB, NASA NAS5-02099)
2. THB_L2_ESA, THEMIS-B (P1/ARTEMIS-P1): Electrostatic Analyzer (ESA): Electron/Ion Ground-Calculate

In [163]:
# Download THEMIS-B magnetic field data
N = 2
df = download_and_unpack(datasets, N, time_interval)
# 134 dens 148 vel


=== Variables for THB_L2_ESA ===
Time Interval: {'Start': '2007-03-07T00:00:00.000Z', 'End': '2026-04-01T00:00:00.000Z'}
0. thb_peif_data_quality, ESA Full Mode Ion Moment Data Quality (0: Good data, non-zero: Data may not be suitable, see: http://themis.ssl.berkeley.edu/esa_flag.shtml.)  [units: None]
1. thb_peif_densityQ, (Good data only) ESA Full (High Angular Resolution, few minute time res.) Mode, Ion Density  [units: None]
2. thb_peif_avgtempQ, -----Ion Average Temperature  [units: None]
3. thb_peif_vthermalQ, -----Ion Thermal Velocity  [units: None]
4. thb_peif_sc_potQ, -----SC Potential (Same time array as Full Ion ESA data)  [units: None]
5. thb_peif_en_efluxQ, -----Ion Energy Flux spectrogram w/  [units: None]
6. thb_peif_t3Q, -----Ion Diagonalized Temperature (Tprp1, Tprp2, Tpar)  [units: None]
7. thb_peif_magt3Q, -----Temperature, Field Aligned (TprpFA1, TprpFA2, TparFA)  [units: None]
8. thb_peif_ptensQ, -----Ion Pressure Tensor  [units: None]
9. thb_peif_mftensQ, -----Io

In [ ]:
# Download THEMIS-B density and velocity data
THEMIS_ion= df.filter(["thb_peir_epoch","thb_peir_density", "thb_peir_velocity_gse"], axis=1)
THEMIS_ion = THEMIS_ion.set_index("thb_peir_epoch")
THEMIS_ion.dropna(inplace=True)
# распаковка списка bx by bz в отдельные столбцы
THEMIS_ion[["v_X_GSE", "v_Y_GSE", "v_Z_GSE"]] = pd.DataFrame(
    THEMIS_ion["thb_peir_velocity_gse"].tolist(), index=THEMIS_ion.index
)
THEMIS_ion = THEMIS_ion.drop(["thb_peir_velocity_gse"], axis=1)
THEMIS_ion["v"] = THEMIS_ion.apply(
    lambda y: np.sqrt(y["v_X_GSE"] ** 2 + y["v_Y_GSE"] ** 2 + y["v_Z_GSE"] ** 2), axis=1
)

THEMIS_ion.rename(
    columns={
        "thb_peir_density": "N_p",
    },
    inplace=True,
)
# THEMIS_ion = THEMIS_ion.drop(["thb_fgs_btotal"], axis=1)
# фильтр плохих данных
# THEMIS_ion = THEMIS_ion.mask((THEMIS_ion["v"] > 1e5) | (THEMIS_ion["v"] < 1e-4))
# THEMIS_ion = THEMIS_ion.mask((THEMIS_ion["v_X_GSE"] > 1e5) | (THEMIS_ion["v_X_GSE"] < 1e-4))
# THEMIS_ion = THEMIS_ion.mask((THEMIS_ion["v_Y_GSE"] > 1e5) | (THEMIS_ion["v_Y_GSE"] < 1e-4))
# THEMIS_ion = THEMIS_ion.mask((THEMIS_ion["v_Z_GSE"] > 1e5) | (THEMIS_ion["v_Z_GSE"] < 1e-4))
THEMIS_ion = clamp_bad_values(THEMIS_ion)
THEMIS_ion.info()
# усреднение 1 час начиная с 00
# df_ion_1h_edit=df_ion_1h.resample('1h').mean()

THEMIS_ion.plot()

In [ ]:
THEMIS_ion

## Magnetic Field

In [ ]:
# List available datasets for THEMIS magnetic field
datasets = list_datasets("THEMIS", "Magnetic Fields (space)")

In [ ]:
# Download THEMIS-B magnetic field data
N = 7
df = download_and_unpack(datasets, N, time_interval)

In [ ]:
# магнитное поле 1 min
THEMIS_MFI = df.filter(["thb_fgs_epoch", "thb_fgs_btotal", "thb_fgs_gse"], axis=1)
THEMIS_MFI = THEMIS_MFI.set_index("thb_fgs_epoch")
THEMIS_MFI.dropna(inplace=True)
# распаковка списка bx by bz в отдельные столбцы
THEMIS_MFI[["B_X_GSE", "B_Y_GSE", "B_Z_GSE"]] = pd.DataFrame(
    THEMIS_MFI["thb_fgs_gse"].tolist(), index=THEMIS_MFI.index
)
THEMIS_MFI = THEMIS_MFI.drop(["thb_fgs_gse"], axis=1)
THEMIS_MFI["B"] = THEMIS_MFI.thb_fgs_btotal
THEMIS_MFI = THEMIS_MFI.drop(["thb_fgs_btotal"], axis=1)
# фильтр плохих данных
# THEMIS_MFI = THEMIS_MFI.mask((THEMIS_MFI["B"] > 1e5) | (THEMIS_MFI["B"] < 1e-4))
# THEMIS_MFI = THEMIS_MFI.mask((THEMIS_MFI["B_X_GSE"] > 1e5) | (THEMIS_MFI["B_X_GSE"] < 1e-4))
# THEMIS_MFI = THEMIS_MFI.mask((THEMIS_MFI["B_Y_GSE"] > 1e5) | (THEMIS_MFI["B_Y_GSE"] < 1e-4))
# THEMIS_MFI = THEMIS_MFI.mask((THEMIS_MFI["B_Z_GSE"] > 1e5) | (THEMIS_MFI["B_Z_GSE"] < 1e-4))
THEMIS_MFI = clamp_bad_values(THEMIS_MFI)
THEMIS_MFI.info()
# усреднение 1 час начиная с 00
# df_MFI_1h_edit=df_MFI_1h.resample('1h').mean()

THEMIS_MFI.plot()

## Coords

In [ ]:
# List available datasets for THEMIS magnetic field
datasets = list_datasets("THEMIS", "Ephemeris/Attitude/Ancillary")

In [ ]:
# Download THEMIS-B magnetic field data
N = 7
df = download_and_unpack(datasets, N, time_interval)

In [ ]:
# магнитное поле 1 min
Themis_B = df.filter(["Epoch", "XYZ_GSE"], axis=1)
Themis_B = Themis_B.set_index("Epoch")
Themis_B.dropna(inplace=True)
# распаковка списка bx by bz в отдельные столбцы
Themis_B[["X_GSE", "Y_GSE", "Z_GSE"]] = pd.DataFrame(
    Themis_B["XYZ_GSE"].tolist(), index=Themis_B.index
)
Themis_B = Themis_B.drop(["XYZ_GSE"], axis=1)
Themis_B = coerce_distance_cols_to_re(
    Themis_B,
    ["X_GSE", "Y_GSE", "Z_GSE"],
    sat_name="THEMIS_B",
    unit_hint=LAST_VAR_UNITS.get("XYZ_GSE"),
)
Themis_B.info()
# усреднение 1 час начиная с 00
# df_GSE_1h_edit=df_GSE_1h.resample('1h').mean()

## Plot and Export

In [160]:
mfi = THEMIS_MFI.copy()
sw = THEMIS_ion.copy()
coord = Themis_B.copy()

# Normalize index names and to UTC datetime index
for df in (mfi, sw, coord):
    df.index = pd.to_datetime(df.index, utc=True)
    df.index.name = "Epoch"

# Replace CDAS fill values (e.g., -1e31) with NaN
mfi = clamp_bad_values(mfi)
coord = clamp_bad_values(coord)

# Pick a "unified" epoch grid — here we use MFI timestamps
# You can swap to sw.index or coord.index if preferred.
epoch = mfi.index

# Nearest-time merge with tolerance (e.g., 2 minutes)
tol = pd.Timedelta("2min")

mfi_u = mfi.reset_index().sort_values("Epoch")
# sw_u = sw.reset_index().sort_values("Epoch")
coord_u = coord.reset_index().sort_values("Epoch")

# merged = pd.merge_asof(
#     mfi_u,
#     sw_u,
#     on="Epoch",
#     direction="nearest",
#     tolerance=tol,
#     suffixes=("_mfi", "_sw"),
# )
merged = pd.merge_asof(mfi_u, coord_u, on="Epoch", direction="nearest", tolerance=tol)

merged = merged.set_index("Epoch").sort_index()

mfi_cols = list(mfi.columns)
sw_cols = list(sw.columns)

merged = merged.dropna(subset=mfi_cols, how="all")

In [161]:
save_parquet(merged, "THEMIS_B", tShock)

PosixPath('Data/2022-11-24 19-10-00/THEMIS_B.parquet')

# THEMIS-C

## Magnetic Field

In [ ]:
# List available datasets for THEMIS magnetic field
datasets = list_datasets("THEMIS", "Magnetic Fields (space)")

In [ ]:
# Download THEMIS-C magnetic field data
N = 12
df = download_and_unpack(datasets, N, time_interval)

In [ ]:
# магнитное поле 1 min
THEMIS_C_MFI = df.filter(["thc_fgs_epoch", "thc_fgs_btotal", "thc_fgs_gse"], axis=1)
THEMIS_C_MFI = THEMIS_C_MFI.set_index("thc_fgs_epoch")
THEMIS_C_MFI.dropna(inplace=True)
# распаковка списка bx by bz в отдельные столбцы
THEMIS_C_MFI[["B_X_GSE", "B_Y_GSE", "B_Z_GSE"]] = pd.DataFrame(
    THEMIS_C_MFI["thc_fgs_gse"].tolist(), index=THEMIS_C_MFI.index
)
THEMIS_C_MFI = THEMIS_C_MFI.drop(["thc_fgs_gse"], axis=1)
THEMIS_C_MFI["B"] = THEMIS_C_MFI.thc_fgs_btotal
THEMIS_C_MFI = THEMIS_C_MFI.drop(["thc_fgs_btotal"], axis=1)
# фильтр плохих данных
# THEMIS_C_MFI = THEMIS_C_MFI.mask((THEMIS_C_MFI["B"] > 1e5) | (THEMIS_C_MFI["B"] < 1e-4))
# THEMIS_C_MFI = THEMIS_C_MFI.mask((THEMIS_C_MFI["B_X_GSE"] > 1e5) | (THEMIS_C_MFI["B_X_GSE"] < 1e-4))
# THEMIS_C_MFI = THEMIS_C_MFI.mask((THEMIS_C_MFI["B_Y_GSE"] > 1e5) | (THEMIS_C_MFI["B_Y_GSE"] < 1e-4))
# THEMIS_C_MFI = THEMIS_C_MFI.mask((THEMIS_C_MFI["B_Z_GSE"] > 1e5) | (THEMIS_C_MFI["B_Z_GSE"] < 1e-4))
THEMIS_C_MFI = clamp_bad_values(THEMIS_C_MFI)
THEMIS_C_MFI.info()

THEMIS_C_MFI.plot()

## Coords

In [ ]:
# List available datasets for THEMIS ephemeris
datasets = list_datasets("THEMIS", "Ephemeris/Attitude/Ancillary")

In [ ]:
# Download THEMIS-C ephemeris data
N = 9
df = download_and_unpack(datasets, N, time_interval)

In [ ]:
df

In [ ]:
# магнитное поле 1 min
Themis_C = df.filter(["thc_state_epoch", "thc_pos_gse"], axis=1)
Themis_C = Themis_C.set_index("thc_state_epoch")
Themis_C.rename(columns={"thc_state_epoch": "Epoch"}, inplace=True)
Themis_C.dropna(inplace=True)
# распаковка списка bx by bz в отдельные столбцы
Themis_C[["X_GSE", "Y_GSE", "Z_GSE"]] = pd.DataFrame(
    Themis_C["thc_pos_gse"].tolist(), index=Themis_C.index
)
Themis_C = Themis_C.drop(["thc_pos_gse"], axis=1)
Themis_C = clamp_bad_values(Themis_C)
Themis_C = coerce_distance_cols_to_re(
    Themis_C,
    ["X_GSE", "Y_GSE", "Z_GSE"],
    sat_name="THEMIS_C",
    unit_hint=LAST_VAR_UNITS.get("thc_pos_gse"),
)
Themis_C.info()

## Plot and Export

In [ ]:
mfi = THEMIS_MFI.copy()
# sw = Ace_SW.copy()
coord = Themis_B.copy()

# Normalize index names and to UTC datetime index
for df in (mfi, sw, coord):
    df.index = pd.to_datetime(df.index, utc=True)
    df.index.name = "Epoch"

# Replace CDAS fill values (e.g., -1e31) with NaN
mfi = clamp_bad_values(mfi)
coord = clamp_bad_values(coord)

# Pick a "unified" epoch grid — here we use MFI timestamps
# You can swap to sw.index or coord.index if preferred.
epoch = mfi.index

# Nearest-time merge with tolerance (e.g., 2 minutes)
tol = pd.Timedelta("2min")

mfi_u = mfi.reset_index().sort_values("Epoch")
# sw_u = sw.reset_index().sort_values("Epoch")
coord_u = coord.reset_index().sort_values("Epoch")

# merged = pd.merge_asof(
#     mfi_u,
#     sw_u,
#     on="Epoch",
#     direction="nearest",
#     tolerance=tol,
#     suffixes=("_mfi", "_sw"),
# )
merged = pd.merge_asof(mfi_u, coord_u, on="Epoch", direction="nearest", tolerance=tol)

merged = merged.set_index("Epoch").sort_index()

mfi_cols = list(mfi.columns)
sw_cols = list(sw.columns)

merged = merged.dropna(subset=mfi_cols, how="all")

In [ ]:
save_parquet(merged, "THEMIS_C", tShock)

# MMS

## Magnetic Field

In [ ]:
datasets = list_datasets("MMS", "Magnetic Fields (space)")

In [ ]:
# Download THEMIS-C ephemeris data
N = 3
df = download_and_unpack(datasets, N, time_interval)
df

In [ ]:
df["mms1_fgm_r_gse_srvy_l2"]

In [ ]:
# магнитное поле 1 min
MMS_MFI = df.filter(
    ["Epoch", "mms1_fgm_b_gse_srvy_l2_clean", "mms1_fgm_r_gse_srvy_l2"], axis=1
)
MMS_MFI = MMS_MFI.set_index("Epoch")

# start by dropping rows with missing magnetic vector
MMS_MFI.dropna(subset="mms1_fgm_b_gse_srvy_l2_clean", inplace=True)

# распаковка списка bx by bz в отдельные столбцы
MMS_MFI[["B_X_GSE", "B_Y_GSE", "B_Z_GSE", "B"]] = pd.DataFrame(
    MMS_MFI["mms1_fgm_b_gse_srvy_l2_clean"].tolist(), index=MMS_MFI.index
)
MMS_MFI = MMS_MFI.drop(["mms1_fgm_b_gse_srvy_l2_clean"], axis=1)

# faster safe-unpack for position vectors:
# prefill NaNs and only assign rows that are valid vec4 values.
import numpy as np

_pos_raw = MMS_MFI["mms1_fgm_r_gse_srvy_l2"].to_numpy(dtype=object)
_coords_np = np.full((len(_pos_raw), 4), np.nan, dtype=float)
_valid_idx = []
_valid_rows = []

for i, v in enumerate(_pos_raw):
    if isinstance(v, (list, tuple, np.ndarray)) and len(v) == 4:
        try:
            _valid_rows.append([float(v[0]), float(v[1]), float(v[2]), float(v[3])])
            _valid_idx.append(i)
        except (TypeError, ValueError):
            pass

if _valid_idx:
    _coords_np[np.array(_valid_idx, dtype=int)] = np.asarray(_valid_rows, dtype=float)

coords = pd.DataFrame(
    _coords_np,
    index=MMS_MFI.index,
    columns=["X_GSE", "Y_GSE", "Z_GSE", "R"],
)

MMS_MFI = MMS_MFI.join(coords)
MMS_MFI = MMS_MFI.drop(["mms1_fgm_r_gse_srvy_l2"], axis=1)
MMS_MFI = coerce_distance_cols_to_re(
    MMS_MFI,
    ["X_GSE", "Y_GSE", "Z_GSE"],
    sat_name="MMS1",
    unit_hint=LAST_VAR_UNITS.get("mms1_fgm_r_gse_srvy_l2"),
    default_unit="km",
)
MMS_MFI = MMS_MFI.drop(["R"], axis=1)
MMS_MFI = clamp_bad_values(MMS_MFI)
MMS_MFI.plot()
## Magnetic Field

## Solar wind

In [ ]:
datasets = list_datasets("MMS", "Plasma and Solar Wind")

## Plot and Export

In [ ]:
save_parquet(MMS_MFI, "MMS1", tShock)